# UPR Bootstrap — Sync Latest `upr/` Code from GitHub

**Run once per Colab session before Notebooks 01–04.**

- Clones the GitHub repo to `/content/upr_tmp` (ephemeral, just for code)
- Copies the latest `upr/` package into your Drive project directory
- Drive keeps the checkpoint & results; GitHub keeps the code

In [ ]:
import os
import sys
import shutil
import subprocess
import importlib

GITHUB_REPO = 'https://github.com/chiragrohit/UniversalPrecisionRuntime.git'
TMP_CLONE   = '/content/upr_tmp'

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
os.makedirs(PROJECT_DIR, exist_ok=True)

# 1. Clone repo to ephemeral /content/ (not Drive)
print('Cloning latest code from GitHub...')
if os.path.exists(TMP_CLONE):
    shutil.rmtree(TMP_CLONE)

result = subprocess.run(
    ['git', 'clone', '--depth=1', GITHUB_REPO, TMP_CLONE],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('git clone failed:', result.stderr)
else:
    print('Clone successful.')

# 2. Copy upr/ from clone → Drive project dir (overwrites old files)
src_upr  = os.path.join(TMP_CLONE, 'upr')
dest_upr = os.path.join(PROJECT_DIR, 'upr')

if os.path.exists(src_upr):
    shutil.copytree(src_upr, dest_upr, dirs_exist_ok=True)
    print(f'Copied upr/ → {dest_upr}')
    for f in os.listdir(dest_upr):
        print(f'  ✓ upr/{f}')
else:
    print('ERROR: upr/ not found in cloned repo.')

# 3. Cleanup temp clone
shutil.rmtree(TMP_CLONE, ignore_errors=True)

# 4. Verify import
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

import upr
importlib.reload(upr)
upr.set_seed(42)
print(f'\nupr {upr.__version__} imported successfully — ready to run Notebooks 01 → 04.')